# 🧪 Experimento: [NOME DO SEU EXPERIMENTO]

**Data:** 2024-06-24  
**Objetivo:** [Descreva o que você quer testar]  
**Baseline:** mAP50=0.9262, mAP50-95=0.7663  

---

## Hipótese

[O que você espera que melhore?]

## Método

[Como você vai testar?]

## Setup

In [ ]:
from pathlib import Path
import sys
import time

# Adicionar src ao path
sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

from ultralytics import YOLO
import torch

# Imports locais
import src.config as cfg
from src.training_logger import TrainingLogger

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Treino

In [ ]:
# Inicializar logger
logger = TrainingLogger(log_dir="../../training_logs")

# Configuração
MODEL_SIZE = "s"  # MUDE PARA SEU EXPERIMENTO
EPOCHS = 100
BATCH_SIZE = 16
IMGSZ = 768

print(f"📦 Modelo: YOLO26{MODEL_SIZE}")
print(f"🔢 Épocas: {EPOCHS}")
print(f"📊 Batch Size: {BATCH_SIZE}")
print(f"🖼️  Tamanho da Imagem: {IMGSZ}")

In [ ]:
# Carregar modelo
model = YOLO(f"yolo26{MODEL_SIZE}.pt")

# Registrar tempo
start_time = time.time()

# Treinar
results = model.train(
    data=str(cfg.DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
    # ADICIONE SEUS PARÂMETROS CUSTOMIZADOS AQUI
)

train_time = time.time() - start_time
print(f"\n✅ Treinamento concluído em {train_time/3600:.2f} horas")

## Validação

In [ ]:
# Validar modelo
metrics = model.val(workers=0)

print(f"\n📊 Resultados:")
print(f"   mAP50:    {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")

# Comparar com baseline
baseline_map50 = 0.9262
baseline_map50_95 = 0.7663

diff_map50 = metrics.box.map50 - baseline_map50
diff_map50_95 = metrics.box.map - baseline_map50_95

print(f"\n📈 Comparação com Baseline:")
print(f"   mAP50: {diff_map50:+.4f} ({diff_map50/baseline_map50*100:+.2f}%)")
print(f"   mAP50-95: {diff_map50_95:+.4f} ({diff_map50_95/baseline_map50_95*100:+.2f}%)")

## 📝 Registrar Treino

In [ ]:
# Registrar no logger
train_id, record = logger.log_training(
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    imgsz=IMGSZ,
    map50=metrics.box.map50,
    map50_95=metrics.box.map,
    train_time=train_time,
    notes="[ADICIONE AQUI SUAS NOTAS SOBRE O EXPERIMENTO]"
)

print(f"\n✅ Treino registrado com ID: {train_id}")
print(f"   Salvo em: training_logs/training_history.json")

## 📊 Histórico Completo

In [ ]:
# Ver todos os treinos
logger.print_summary()

## 🧪 Teste com Threshold (Opcional)

In [ ]:
# Se quiser testar diferentes thresholds
from src.test_thresholds import test_thresholds

# Encontrar o melhor modelo treinado
best_pt = Path("runs/detect/train-X/weights/best.pt")  # MUDE X para seu train

# Testar thresholds
results = test_thresholds(
    model_path=str(best_pt),
    image_path="../testes/spiders.jpg",
    thresholds=[0.01, 0.05, 0.1, 0.15, 0.2, 0.25],
    save_dir="threshold_tests"
)

## 💾 Salvar Modelo (se bom)

In [ ]:
# Se o modelo ficou bom, salve em models/experiments/
from pathlib import Path
import shutil

# MUDE ESSES VALORES
EXPERIMENT_NAME = "exp_XXX_nome_do_experimento"
BEST_PT = Path("runs/detect/train-X/weights/best.pt")  # MUDE X

output_dir = Path(f"../../models/experiments/{EXPERIMENT_NAME}")
output_dir.mkdir(parents=True, exist_ok=True)

# Copiar modelo
shutil.copy(BEST_PT, output_dir / "best.pt")

# Salvar notas
notes = f"""YOLO26{MODEL_SIZE}
Épocas: {EPOCHS}
mAP50: {metrics.box.map50:.4f}
mAP50-95: {metrics.box.map:.4f}

[ADICIONE MAIS NOTAS AQUI]"""

with open(output_dir / "notes.txt", "w", encoding="utf-8") as f:
    f.write(notes)

print(f"✅ Modelo salvo em: {output_dir.absolute()}")

## 📋 Conclusões

[Escreva aqui o que você aprendeu com esse experimento]